# 📐 Modelado de Datos — Modelo Normalizado INUMET
**Materia:** Herramientas de Software para Big Data  

**Modelo:** Normalizado  
**Fuente:** `/rfn/obligatorio/`  
**Destino:** `/models/obligatorio/`

**Tablas del modelo:**
```
dim_estaciones     — datos de cada estacion meteorologica
dim_tiempo         — dimension temporal con atributos derivados
fact_temperatura   — mediciones de temperatura del aire
fact_viento        — mediciones de intensidad y direccion del viento
fact_precipitacion — mediciones de precipitacion horaria
fact_humedad       — mediciones de humedad relativa
fact_presion       — mediciones de presion atmosferica
fact_insolacion    — mediciones de horas de insolacion solar
```

## 1. Inicializacion de Spark

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, year, month, dayofmonth, hour,
    when, monotonically_increasing_id,
    lit, concat, lpad
)
from pyspark.sql.types import *

spark = SparkSession.builder \
    .appName("INUMET_Modelado") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
print(f"Spark {spark.version} listo")

## 2. Carga desde /rfn

In [ ]:
RFN = "hdfs://localhost:9000/rfn/obligatorio"

df_temp    = spark.read.parquet(f"{RFN}/temperatura")
df_viento  = spark.read.parquet(f"{RFN}/viento")
df_lluvia  = spark.read.parquet(f"{RFN}/precipitacion")
df_humedad = spark.read.parquet(f"{RFN}/humedad")
df_presion = spark.read.parquet(f"{RFN}/presion")
df_helio   = spark.read.parquet(f"{RFN}/heliofania")

print("Tablas cargadas desde /rfn:")
for nombre, df in [("temperatura", df_temp), ("viento", df_viento),
                   ("precipitacion", df_lluvia), ("humedad", df_humedad),
                   ("presion", df_presion), ("heliofania", df_helio)]:
    print(f"  {nombre:<15}: {df.count():>10,} registros")

## 3. dim_estaciones
Construida manualmente con los datos conocidos de cada estacion.

In [ ]:
estaciones_data = [
    ("Aeropuerto Melilla G3", "Montevideo",  "costera",  -34.8335, -56.0303),
    ("Artigas G3",            "Artigas",     "interior", -30.4000, -56.5000),
    ("Colonia G3",            "Colonia",     "costera",  -34.4622, -57.8408),
    ("Mercedes G3",           "Soriano",     "interior", -33.2500, -58.0833),
    ("Paso de los Toros G3",  "Tacuarembo",  "interior", -32.8167, -56.5167),
    ("Rocha G3",              "Rocha",       "costera",  -34.4833, -54.3333),
    ("Salto G3",              "Salto",       "interior", -31.3833, -57.9667),
]

schema_est = StructType([
    StructField("estacion_id",  StringType(),  False),
    StructField("departamento", StringType(),  False),
    StructField("zona",         StringType(),  False),
    StructField("latitud",      DoubleType(),  True),
    StructField("longitud",     DoubleType(),  True),
])

dim_estaciones = spark.createDataFrame(estaciones_data, schema=schema_est)

print("dim_estaciones:")
dim_estaciones.show(truncate=False)

## 4. dim_tiempo
Se construye a partir de todas las fechas unicas que existen en los datos.
Atributos derivados: anio, mes, dia, hora, estacion_anio.

In [ ]:
# Unir todas las fechas unicas de todas las tablas
fechas = df_temp.select("fecha") \
    .union(df_viento.select("fecha")) \
    .union(df_lluvia.select("fecha")) \
    .union(df_humedad.select("fecha")) \
    .union(df_presion.select("fecha")) \
    .union(df_helio.select("fecha")) \
    .distinct()

# Agregar atributos temporales
dim_tiempo = fechas \
    .withColumn("anio",  year(col("fecha"))) \
    .withColumn("mes",   month(col("fecha"))) \
    .withColumn("dia",   dayofmonth(col("fecha"))) \
    .withColumn("hora",  hour(col("fecha"))) \
    .withColumn("estacion_anio",
        when((col("mes") >= 12) | (col("mes") <= 2),  "verano")
        .when((col("mes") >= 3)  & (col("mes") <= 5),  "otonio")
        .when((col("mes") >= 6)  & (col("mes") <= 8),  "invierno")
        .otherwise("primavera")
    ) \
    .withColumn("nombre_mes",
        when(col("mes") == 1,  "Enero")
        .when(col("mes") == 2,  "Febrero")
        .when(col("mes") == 3,  "Marzo")
        .when(col("mes") == 4,  "Abril")
        .when(col("mes") == 5,  "Mayo")
        .when(col("mes") == 6,  "Junio")
        .when(col("mes") == 7,  "Julio")
        .when(col("mes") == 8,  "Agosto")
        .when(col("mes") == 9,  "Septiembre")
        .when(col("mes") == 10, "Octubre")
        .when(col("mes") == 11, "Noviembre")
        .otherwise("Diciembre")
    ) \
    .orderBy("fecha")

print(f"dim_tiempo: {dim_tiempo.count():,} registros unicos")
dim_tiempo.show(5, truncate=False)

## 5. Tablas de hechos
Cada tabla de hechos conserva `fecha` y `estacion_id` como claves foraneas hacia `dim_tiempo` y `dim_estaciones`.

In [ ]:
# fact_temperatura
fact_temperatura = df_temp.select(
    col("fecha"),
    col("estacion_id"),
    col("temp_aire")
)

# fact_viento
fact_viento = df_viento.select(
    col("fecha"),
    col("estacion_id"),
    col("int_viento"),
    col("dir_viento")
)

# fact_precipitacion
fact_precipitacion = df_lluvia.select(
    col("fecha"),
    col("estacion_id"),
    col("precip_horario")
)

# fact_humedad
fact_humedad = df_humedad.select(
    col("fecha"),
    col("estacion_id"),
    col("hum_relativa")
)

# fact_presion
fact_presion = df_presion.select(
    col("fecha"),
    col("estacion_id"),
    col("pres_atm_mar")
)

# fact_insolacion (renombramos heliofania -> horas_insolacion)
fact_insolacion = df_helio.select(
    col("fecha"),
    col("estacion_id"),
    col("heliofania").alias("horas_insolacion")
)

facts = {
    "fact_temperatura":   fact_temperatura,
    "fact_viento":        fact_viento,
    "fact_precipitacion": fact_precipitacion,
    "fact_humedad":       fact_humedad,
    "fact_presion":       fact_presion,
    "fact_insolacion":    fact_insolacion,
}

print("Tablas de hechos:")
for nombre, df in facts.items():
    print(f"  {nombre:<22}: {df.count():>10,} registros | columnas: {df.columns}")

## 6. Verificacion de integridad referencial
Comprobamos que todos los `estacion_id` de las tablas de hechos existen en `dim_estaciones`.

In [ ]:
estaciones_validas = dim_estaciones.select("estacion_id")

print("Verificacion de integridad referencial — estacion_id:\n")
for nombre, df in facts.items():
    huerfanos = df.select("estacion_id").distinct() \
        .subtract(estaciones_validas) \
        .count()
    icono = "✅" if huerfanos == 0 else "❌"
    print(f"  {icono} {nombre:<25} estaciones sin match: {huerfanos}")

## 7. Vista previa del modelo completo

In [ ]:
print("=== dim_estaciones ===")
dim_estaciones.show(truncate=False)

print("\n=== dim_tiempo (muestra) ===")
dim_tiempo.show(5, truncate=False)

print("\n=== fact_temperatura (muestra) ===")
fact_temperatura.show(5, truncate=False)

print("\n=== fact_insolacion (muestra) ===")
fact_insolacion.show(5, truncate=False)

## 8. Guardado en /models
Las tablas del modelo normalizado se guardan en `/models/obligatorio/`  
en formato Parquet. A esta ubicacion apuntaran las tablas externas de Hive.

In [ ]:
MODELS = "hdfs://localhost:9000/models/obligatorio"

# Dimensiones
dim_estaciones.write.mode("overwrite").parquet(f"{MODELS}/dim_estaciones")
dim_tiempo.write.mode("overwrite").parquet(f"{MODELS}/dim_tiempo")

# Hechos
fact_temperatura.write.mode("overwrite").parquet(f"{MODELS}/fact_temperatura")
fact_viento.write.mode("overwrite").parquet(f"{MODELS}/fact_viento")
fact_precipitacion.write.mode("overwrite").parquet(f"{MODELS}/fact_precipitacion")
fact_humedad.write.mode("overwrite").parquet(f"{MODELS}/fact_humedad")
fact_presion.write.mode("overwrite").parquet(f"{MODELS}/fact_presion")
fact_insolacion.write.mode("overwrite").parquet(f"{MODELS}/fact_insolacion")

print("Modelo guardado en /models/obligatorio:")
print("  ✅ dim_estaciones")
print("  ✅ dim_tiempo")
print("  ✅ fact_temperatura")
print("  ✅ fact_viento")
print("  ✅ fact_precipitacion")
print("  ✅ fact_humedad")
print("  ✅ fact_presion")
print("  ✅ fact_insolacion")

## 9. Verificacion final — estructura en /models

In [ ]:
# Releer desde /models para confirmar que se guardaron correctamente
print("Verificacion de escritura en /models:\n")
tablas_model = [
    "dim_estaciones", "dim_tiempo",
    "fact_temperatura", "fact_viento", "fact_precipitacion",
    "fact_humedad", "fact_presion", "fact_insolacion"
]

print(f"{'Tabla':<25} {'Registros':>12} {'Columnas'}")
print("-"*60)
for tabla in tablas_model:
    df = spark.read.parquet(f"{MODELS}/{tabla}")
    print(f"  {tabla:<23} {df.count():>12,}  {df.columns}")

In [ ]:
spark.stop()
print("Sesion Spark cerrada.")